# YOLO v3 Object Detection

YOLO (You Only Look Once) is a real-time object detection algorithm that uses convolutional neural networks to predict bounding boxes and class probabilities simultaneously. Unlike two-stage detectors, YOLO processes the full image in a single forward pass, making it extremely fast.

This notebook implements **YOLOv3** with a Darknet-53 backbone for detecting household and general objects from the COCO dataset (80 classes).

> **Note:** This implementation uses the TensorFlow 1.x API. Ensure you are running with `tf.compat.v1` or TensorFlow 1.x.

## 1. Dependencies

In [1]:
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

import numpy as np
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display
from seaborn import color_palette
import cv2

print('TensorFlow version:', tf.__version__)

ModuleNotFoundError: No module named 'tensorflow'

## 2. Model Hyperparameters

- **Batch normalization** uses momentum for computing moving average/variance during inference.
- **Leaky ReLU** prevents "dying neurons" by allowing a small negative gradient.
- **Anchors** are bounding box priors computed via k-means on the COCO dataset.
- **Model size** is the fixed input resolution: 416×416 pixels.

In [2]:
_BATCH_NORM_DECAY = 0.9
_BATCH_NORM_EPSILON = 1e-05
_LEAKY_RELU = 0.1
_ANCHORS = [
    (10, 13),  (16, 30),   (33, 23),    # small objects
    (30, 61),  (62, 45),   (59, 119),   # medium objects
    (116, 90), (156, 198), (373, 326)   # large objects
]
_MODEL_SIZE = (416, 416)

## 3. Model Definition

### 3.1 Batch Normalization & Fixed Padding

In [3]:
def batch_norm(inputs, training, data_format):
    """Performs batch normalization using a standard set of parameters."""
    return tf.layers.batch_normalization(
        inputs=inputs,
        axis=1 if data_format == 'channels_first' else 3,
        momentum=_BATCH_NORM_DECAY,
        epsilon=_BATCH_NORM_EPSILON,
        scale=True,
        training=training)


def fixed_padding(inputs, kernel_size, data_format):
    """Pads the input along spatial dimensions independently of input size.

    Args:
        inputs: Tensor input to be padded.
        kernel_size: The kernel to be used in the conv2d or max_pool2d.
        data_format: The input format.
    Returns:
        A tensor with the same format as the input.
    """
    pad_total = kernel_size - 1
    pad_beg = pad_total // 2
    pad_end = pad_total - pad_beg

    if data_format == 'channels_first':
        padded_inputs = tf.pad(inputs, [[0, 0], [0, 0],
                                        [pad_beg, pad_end],
                                        [pad_beg, pad_end]])
    else:
        padded_inputs = tf.pad(inputs, [[0, 0], [pad_beg, pad_end],
                                        [pad_beg, pad_end], [0, 0]])
    return padded_inputs


def conv2d_fixed_padding(inputs, filters, kernel_size, data_format, strides=1):
    """Strided 2-D convolution with explicit padding."""
    if strides > 1:
        inputs = fixed_padding(inputs, kernel_size, data_format)

    return tf.layers.conv2d(
        inputs=inputs,
        filters=filters,
        kernel_size=kernel_size,
        strides=strides,
        padding=('SAME' if strides == 1 else 'VALID'),
        use_bias=False,
        data_format=data_format)

### 3.2 Feature Extraction: Darknet-53

Darknet-53 is the backbone network pretrained on ImageNet. It uses residual (shortcut) connections like ResNet. We extract feature maps at three scales (`route1`, `route2`, and the final output) for multi-scale detection.

In [4]:
def darknet53_residual_block(inputs, filters, training, data_format, strides=1):
    """Creates a residual block for Darknet."""
    shortcut = inputs

    inputs = conv2d_fixed_padding(inputs, filters=filters, kernel_size=1,
                                  strides=strides, data_format=data_format)
    inputs = batch_norm(inputs, training=training, data_format=data_format)
    inputs = tf.nn.leaky_relu(inputs, alpha=_LEAKY_RELU)

    inputs = conv2d_fixed_padding(inputs, filters=2 * filters, kernel_size=3,
                                  strides=strides, data_format=data_format)
    inputs = batch_norm(inputs, training=training, data_format=data_format)
    inputs = tf.nn.leaky_relu(inputs, alpha=_LEAKY_RELU)

    inputs += shortcut
    return inputs


def darknet53(inputs, training, data_format):
    """Creates Darknet53 model for feature extraction.
    Returns three feature maps at different scales.
    """
    inputs = conv2d_fixed_padding(inputs, filters=32, kernel_size=3,
                                  data_format=data_format)
    inputs = batch_norm(inputs, training=training, data_format=data_format)
    inputs = tf.nn.leaky_relu(inputs, alpha=_LEAKY_RELU)

    inputs = conv2d_fixed_padding(inputs, filters=64, kernel_size=3,
                                  strides=2, data_format=data_format)
    inputs = batch_norm(inputs, training=training, data_format=data_format)
    inputs = tf.nn.leaky_relu(inputs, alpha=_LEAKY_RELU)

    inputs = darknet53_residual_block(inputs, filters=32, training=training,
                                      data_format=data_format)

    inputs = conv2d_fixed_padding(inputs, filters=128, kernel_size=3,
                                  strides=2, data_format=data_format)
    inputs = batch_norm(inputs, training=training, data_format=data_format)
    inputs = tf.nn.leaky_relu(inputs, alpha=_LEAKY_RELU)

    for _ in range(2):
        inputs = darknet53_residual_block(inputs, filters=64, training=training,
                                          data_format=data_format)

    inputs = conv2d_fixed_padding(inputs, filters=256, kernel_size=3,
                                  strides=2, data_format=data_format)
    inputs = batch_norm(inputs, training=training, data_format=data_format)
    inputs = tf.nn.leaky_relu(inputs, alpha=_LEAKY_RELU)

    for _ in range(8):
        inputs = darknet53_residual_block(inputs, filters=128, training=training,
                                          data_format=data_format)

    route1 = inputs  # scale 1: large feature map (small objects)

    inputs = conv2d_fixed_padding(inputs, filters=512, kernel_size=3,
                                  strides=2, data_format=data_format)
    inputs = batch_norm(inputs, training=training, data_format=data_format)
    inputs = tf.nn.leaky_relu(inputs, alpha=_LEAKY_RELU)

    for _ in range(8):
        inputs = darknet53_residual_block(inputs, filters=256, training=training,
                                          data_format=data_format)

    route2 = inputs  # scale 2: medium feature map

    inputs = conv2d_fixed_padding(inputs, filters=1024, kernel_size=3,
                                  strides=2, data_format=data_format)
    inputs = batch_norm(inputs, training=training, data_format=data_format)
    inputs = tf.nn.leaky_relu(inputs, alpha=_LEAKY_RELU)

    for _ in range(4):
        inputs = darknet53_residual_block(inputs, filters=512, training=training,
                                          data_format=data_format)

    return route1, route2, inputs  # scale 3: small feature map (large objects)

### 3.3 YOLO Convolution Block

After Darknet-53, each detection scale passes through 5 convolutional layers (1×1 → 3×3 alternating) before the final detection head.

In [5]:
def yolo_convolution_block(inputs, filters, training, data_format):
    """Creates convolution operations layer used after Darknet."""
    inputs = conv2d_fixed_padding(inputs, filters=filters, kernel_size=1,
                                  data_format=data_format)
    inputs = batch_norm(inputs, training=training, data_format=data_format)
    inputs = tf.nn.leaky_relu(inputs, alpha=_LEAKY_RELU)

    inputs = conv2d_fixed_padding(inputs, filters=2 * filters, kernel_size=3,
                                  data_format=data_format)
    inputs = batch_norm(inputs, training=training, data_format=data_format)
    inputs = tf.nn.leaky_relu(inputs, alpha=_LEAKY_RELU)

    inputs = conv2d_fixed_padding(inputs, filters=filters, kernel_size=1,
                                  data_format=data_format)
    inputs = batch_norm(inputs, training=training, data_format=data_format)
    inputs = tf.nn.leaky_relu(inputs, alpha=_LEAKY_RELU)

    inputs = conv2d_fixed_padding(inputs, filters=2 * filters, kernel_size=3,
                                  data_format=data_format)
    inputs = batch_norm(inputs, training=training, data_format=data_format)
    inputs = tf.nn.leaky_relu(inputs, alpha=_LEAKY_RELU)

    inputs = conv2d_fixed_padding(inputs, filters=filters, kernel_size=1,
                                  data_format=data_format)
    inputs = batch_norm(inputs, training=training, data_format=data_format)
    inputs = tf.nn.leaky_relu(inputs, alpha=_LEAKY_RELU)

    route = inputs  # branch off for next scale

    inputs = conv2d_fixed_padding(inputs, filters=2 * filters, kernel_size=3,
                                  data_format=data_format)
    inputs = batch_norm(inputs, training=training, data_format=data_format)
    inputs = tf.nn.leaky_relu(inputs, alpha=_LEAKY_RELU)

    return route, inputs

### 3.4 Detection Layer

Each detection layer operates at a different scale with 3 anchors. It predicts `n_anchors * (5 + n_classes)` values per grid cell:
- 4 box coordinates (center x, center y, width, height)
- 1 objectness/confidence score
- `n_classes` class probabilities

In [6]:
def yolo_layer(inputs, n_classes, anchors, img_size, data_format):
    """Creates Yolo final detection layer.

    Args:
        inputs: Tensor input.
        n_classes: Number of labels.
        anchors: A list of anchor sizes.
        img_size: The input size of the model.
        data_format: The input format.
    Returns:
        Tensor output with decoded box predictions.
    """
    n_anchors = len(anchors)

    inputs = tf.layers.conv2d(inputs, filters=n_anchors * (5 + n_classes),
                              kernel_size=1, strides=1, use_bias=True,
                              data_format=data_format)

    shape = inputs.get_shape().as_list()
    grid_shape = shape[2:4] if data_format == 'channels_first' else shape[1:3]

    if data_format == 'channels_first':
        inputs = tf.transpose(inputs, [0, 2, 3, 1])

    inputs = tf.reshape(inputs,
                        [-1, n_anchors * grid_shape[0] * grid_shape[1],
                         5 + n_classes])

    strides = (img_size[0] // grid_shape[0], img_size[1] // grid_shape[1])

    box_centers, box_shapes, confidence, classes = \
        tf.split(inputs, [2, 2, 1, n_classes], axis=-1)

    # Build grid offsets
    x = tf.range(grid_shape[0], dtype=tf.float32)
    y = tf.range(grid_shape[1], dtype=tf.float32)
    x_offset, y_offset = tf.meshgrid(x, y)
    x_offset = tf.reshape(x_offset, (-1, 1))
    y_offset = tf.reshape(y_offset, (-1, 1))
    x_y_offset = tf.concat([x_offset, y_offset], axis=-1)
    x_y_offset = tf.tile(x_y_offset, [1, n_anchors])
    x_y_offset = tf.reshape(x_y_offset, [1, -1, 2])

    # Decode box centers
    box_centers = tf.nn.sigmoid(box_centers)
    box_centers = (box_centers + x_y_offset) * strides

    # Decode box dimensions
    anchors = tf.tile(anchors, [grid_shape[0] * grid_shape[1], 1])
    box_shapes = tf.exp(box_shapes) * tf.cast(anchors, tf.float32)

    confidence = tf.nn.sigmoid(confidence)
    classes = tf.nn.sigmoid(classes)

    inputs = tf.concat([box_centers, box_shapes, confidence, classes], axis=-1)
    return inputs

### 3.5 Upsample Layer

Nearest-neighbor upsampling is used to merge coarser feature maps from deeper layers with finer feature maps from earlier layers for multi-scale detection.

In [7]:
def upsample(inputs, out_shape, data_format):
    """Upsamples to `out_shape` using nearest neighbor interpolation."""
    if data_format == 'channels_first':
        inputs = tf.transpose(inputs, [0, 2, 3, 1])
        new_height = out_shape[3]
        new_width = out_shape[2]
    else:
        new_height = out_shape[2]
        new_width = out_shape[1]

    inputs = tf.image.resize_nearest_neighbor(inputs, (new_height, new_width))

    if data_format == 'channels_first':
        inputs = tf.transpose(inputs, [0, 3, 1, 2])

    return inputs

### 3.6 Non-Max Suppression

After detection, we filter out:
1. Boxes with low confidence scores (below `confidence_threshold`)
2. Duplicate boxes for the same object using IoU-based NMS per class

In [8]:
def build_boxes(inputs):
    """Computes top-left and bottom-right corners from center-format boxes."""
    center_x, center_y, width, height, confidence, classes = \
        tf.split(inputs, [1, 1, 1, 1, 1, -1], axis=-1)

    top_left_x = center_x - width / 2
    top_left_y = center_y - height / 2
    bottom_right_x = center_x + width / 2
    bottom_right_y = center_y + height / 2

    boxes = tf.concat([top_left_x, top_left_y,
                       bottom_right_x, bottom_right_y,
                       confidence, classes], axis=-1)
    return boxes


def non_max_suppression(inputs, n_classes, max_output_size,
                        iou_threshold, confidence_threshold):
    """Performs non-max suppression separately for each class.

    Args:
        inputs: Tensor input.
        n_classes: Number of classes.
        max_output_size: Max number of boxes per class.
        iou_threshold: Threshold for IoU overlap.
        confidence_threshold: Minimum confidence to keep a box.
    Returns:
        A list of class-to-boxes dicts for each sample in the batch.
    """
    batch = tf.unstack(inputs)
    boxes_dicts = []

    for boxes in batch:
        boxes = tf.boolean_mask(boxes, boxes[:, 4] > confidence_threshold)
        classes = tf.argmax(boxes[:, 5:], axis=-1)
        classes = tf.expand_dims(tf.cast(classes, tf.float32), axis=-1)
        boxes = tf.concat([boxes[:, :5], classes], axis=-1)

        boxes_dict = dict()
        for cls in range(n_classes):
            mask = tf.equal(boxes[:, 5], cls)
            mask_shape = mask.get_shape()
            if mask_shape.ndims != 0:
                class_boxes = tf.boolean_mask(boxes, mask)
                boxes_coords, boxes_conf_scores, _ = tf.split(
                    class_boxes, [4, 1, -1], axis=-1)
                boxes_conf_scores = tf.reshape(boxes_conf_scores, [-1])
                indices = tf.image.non_max_suppression(
                    boxes_coords, boxes_conf_scores,
                    max_output_size, iou_threshold)
                class_boxes = tf.gather(class_boxes, indices)
                boxes_dict[cls] = class_boxes[:, :5]

        boxes_dicts.append(boxes_dict)

    return boxes_dicts

### 3.7 Full Yolo v3 Model

The full model assembles Darknet-53 + three detection heads at scales 13×13, 26×26, and 52×52. Outputs from all three scales are merged before NMS.

In [9]:
class Yolo_v3:
    """YOLOv3 model for multi-scale object detection."""

    def __init__(self, n_classes, model_size, max_output_size,
                 iou_threshold, confidence_threshold, data_format=None):
        """Initialises the model.

        Args:
            n_classes: Number of class labels.
            model_size: Input size of the model (H, W).
            max_output_size: Max boxes per class after NMS.
            iou_threshold: IoU threshold for NMS.
            confidence_threshold: Min confidence to keep a detection.
            data_format: 'channels_first' (GPU) or 'channels_last' (CPU).
        """
        if not data_format:
            data_format = 'channels_first' if tf.test.is_built_with_cuda() \
                else 'channels_last'

        self.n_classes = n_classes
        self.model_size = model_size
        self.max_output_size = max_output_size
        self.iou_threshold = iou_threshold
        self.confidence_threshold = confidence_threshold
        self.data_format = data_format

    def __call__(self, inputs, training):
        """Runs a forward pass and returns detected boxes.

        Args:
            inputs: A Tensor [batch, H, W, 3] of input images.
            training: Boolean, True during training.
        Returns:
            List of class-to-boxes dicts for each sample.
        """
        with tf.variable_scope('yolo_v3_model'):
            if self.data_format == 'channels_first':
                inputs = tf.transpose(inputs, [0, 3, 1, 2])

            inputs = inputs / 255.0  # normalise to [0, 1]

            route1, route2, inputs = darknet53(
                inputs, training=training, data_format=self.data_format)

            # --- Scale 1: 13x13 (large objects) ---
            route, inputs = yolo_convolution_block(
                inputs, filters=512, training=training,
                data_format=self.data_format)
            detect1 = yolo_layer(inputs, n_classes=self.n_classes,
                                 anchors=_ANCHORS[6:9],
                                 img_size=self.model_size,
                                 data_format=self.data_format)

            # --- Scale 2: 26x26 (medium objects) ---
            inputs = conv2d_fixed_padding(route, filters=256, kernel_size=1,
                                          data_format=self.data_format)
            inputs = batch_norm(inputs, training=training,
                                data_format=self.data_format)
            inputs = tf.nn.leaky_relu(inputs, alpha=_LEAKY_RELU)
            upsample_size = route2.get_shape().as_list()
            inputs = upsample(inputs, out_shape=upsample_size,
                              data_format=self.data_format)
            axis = 1 if self.data_format == 'channels_first' else 3
            inputs = tf.concat([inputs, route2], axis=axis)
            route, inputs = yolo_convolution_block(
                inputs, filters=256, training=training,
                data_format=self.data_format)
            detect2 = yolo_layer(inputs, n_classes=self.n_classes,
                                 anchors=_ANCHORS[3:6],
                                 img_size=self.model_size,
                                 data_format=self.data_format)

            # --- Scale 3: 52x52 (small objects) ---
            inputs = conv2d_fixed_padding(route, filters=128, kernel_size=1,
                                          data_format=self.data_format)
            inputs = batch_norm(inputs, training=training,
                                data_format=self.data_format)
            inputs = tf.nn.leaky_relu(inputs, alpha=_LEAKY_RELU)
            upsample_size = route1.get_shape().as_list()
            inputs = upsample(inputs, out_shape=upsample_size,
                              data_format=self.data_format)
            inputs = tf.concat([inputs, route1], axis=axis)
            route, inputs = yolo_convolution_block(
                inputs, filters=128, training=training,
                data_format=self.data_format)
            detect3 = yolo_layer(inputs, n_classes=self.n_classes,
                                 anchors=_ANCHORS[0:3],
                                 img_size=self.model_size,
                                 data_format=self.data_format)

            # Merge all three scales
            inputs = tf.concat([detect1, detect2, detect3], axis=1)
            inputs = build_boxes(inputs)

            boxes_dicts = non_max_suppression(
                inputs, n_classes=self.n_classes,
                max_output_size=self.max_output_size,
                iou_threshold=self.iou_threshold,
                confidence_threshold=self.confidence_threshold)

            return boxes_dicts

## 4. Utility Functions

In [10]:
def load_images(img_names, model_size):
    """Loads images and resizes them to model_size, returns a 4D NumPy array.

    Args:
        img_names: List of image file paths.
        model_size: (H, W) tuple for resizing.
    Returns:
        NumPy array of shape [N, H, W, 3].
    """
    imgs = []
    for img_name in img_names:
        img = Image.open(img_name).convert('RGB')
        img = img.resize(size=model_size)
        img = np.array(img, dtype=np.float32)
        img = np.expand_dims(img, axis=0)
        imgs.append(img)
    return np.concatenate(imgs)


def load_class_names(file_name):
    """Returns a list of class names read from file_name (one per line)."""
    with open(file_name, 'r') as f:
        class_names = f.read().splitlines()
    return class_names


def draw_boxes(img_names, boxes_dicts, class_names, model_size,
               font_path=None):
    """Draws detected bounding boxes with class labels on images.

    Args:
        img_names: List of input image paths.
        boxes_dicts: List of class-to-boxes dicts (one per image).
        class_names: List of class name strings.
        model_size: (H, W) of model input.
        font_path: Optional path to a .ttf font file.
    """
    colors = ((np.array(color_palette('hls', 80)) * 255)).astype(np.uint8)

    for img_name, boxes_dict in zip(img_names, boxes_dicts):
        img = Image.open(img_name).convert('RGB')
        draw = ImageDraw.Draw(img)

        # Load font or fall back to default
        try:
            font_size = (img.size[0] + img.size[1]) // 100
            font = ImageFont.truetype(
                font=font_path or 'arial.ttf', size=font_size)
        except (IOError, OSError):
            font = ImageFont.load_default()

        resize_factor = (img.size[0] / model_size[0],
                         img.size[1] / model_size[1])

        for cls in range(len(class_names)):
            boxes = boxes_dict.get(cls)
            if boxes is None or np.size(boxes) == 0:
                continue

            color = tuple(colors[cls].tolist())
            for box in boxes:
                xy, confidence = box[:4], box[4]
                xy = [xy[i] * resize_factor[i % 2] for i in range(4)]
                x0, y0 = xy[0], xy[1]
                thickness = (img.size[0] + img.size[1]) // 200
                for t in np.linspace(0, 1, thickness):
                    xy[0], xy[1] = xy[0] + t, xy[1] + t
                    xy[2], xy[3] = xy[2] - t, xy[3] - t
                    draw.rectangle(xy, outline=color)

                label = '{} {:.1f}%'.format(class_names[cls], confidence * 100)
                try:
                    text_bbox = draw.textbbox((x0, y0), label, font=font)
                    text_w = text_bbox[2] - text_bbox[0]
                    text_h = text_bbox[3] - text_bbox[1]
                except AttributeError:
                    text_w, text_h = draw.textsize(label, font=font)

                draw.rectangle(
                    [x0, y0 - text_h, x0 + text_w, y0], fill=color)
                draw.text((x0, y0 - text_h), label, fill='black', font=font)

        display(img)

## 5. Converting Official Weights to TensorFlow Format

Download the official YOLOv3 weights from https://pjreddie.com/media/files/yolov3.weights  
and place them in the `weights/` subdirectory.

Also download the COCO class names file (80 classes):  
https://raw.githubusercontent.com/pjreddie/darknet/master/data/coco.names  
and save it as `weights/coco.names`.

In [11]:
def load_weights(variables, file_name):
    """Reshapes and loads official pretrained YOLOv3 weights from a .weights file.

    Args:
        variables: List of tf.Variable objects to be assigned.
        file_name: Path to the .weights file.
    Returns:
        A list of tf.assign operations.
    """
    with open(file_name, 'rb') as f:
        # Skip header (5 int32 values: major, minor, revision, seen, ...)
        np.fromfile(f, dtype=np.int32, count=5)
        weights = np.fromfile(f, dtype=np.float32)

    assign_ops = []
    ptr = 0

    # --- Darknet-53 backbone: 52 conv+BN blocks ---
    for i in range(52):
        conv_var = variables[5 * i]
        gamma, beta, mean, variance = variables[5 * i + 1:5 * i + 5]
        batch_norm_vars = [beta, gamma, mean, variance]

        for var in batch_norm_vars:
            shape = var.shape.as_list()
            num_params = np.prod(shape)
            var_weights = weights[ptr:ptr + num_params].reshape(shape)
            ptr += num_params
            assign_ops.append(tf.assign(var, var_weights))

        shape = conv_var.shape.as_list()
        num_params = np.prod(shape)
        var_weights = weights[ptr:ptr + num_params].reshape(
            (shape[3], shape[2], shape[0], shape[1]))
        var_weights = np.transpose(var_weights, (2, 3, 1, 0))
        ptr += num_params
        assign_ops.append(tf.assign(conv_var, var_weights))

    # --- YOLO detection head layers ---
    # Layers 7, 15, 23 (per range group) have biases and no batch norm
    ranges = [range(0, 6), range(6, 13), range(13, 20)]
    unnormalized = [6, 13, 20]

    for j in range(3):
        for i in ranges[j]:
            current = 52 * 5 + 5 * i + j * 2
            conv_var = variables[current]
            gamma, beta, mean, variance = variables[current + 1:current + 5]
            batch_norm_vars = [beta, gamma, mean, variance]

            for var in batch_norm_vars:
                shape = var.shape.as_list()
                num_params = np.prod(shape)
                var_weights = weights[ptr:ptr + num_params].reshape(shape)
                ptr += num_params
                assign_ops.append(tf.assign(var, var_weights))

            shape = conv_var.shape.as_list()
            num_params = np.prod(shape)
            var_weights = weights[ptr:ptr + num_params].reshape(
                (shape[3], shape[2], shape[0], shape[1]))
            var_weights = np.transpose(var_weights, (2, 3, 1, 0))
            ptr += num_params
            assign_ops.append(tf.assign(conv_var, var_weights))

        # Load bias for unnormalized layer
        bias = variables[52 * 5 + unnormalized[j] * 5 + j * 2 + 1]
        shape = bias.shape.as_list()
        num_params = np.prod(shape)
        var_weights = weights[ptr:ptr + num_params].reshape(shape)
        ptr += num_params
        assign_ops.append(tf.assign(bias, var_weights))

        conv_var = variables[52 * 5 + unnormalized[j] * 5 + j * 2]
        shape = conv_var.shape.as_list()
        num_params = np.prod(shape)
        var_weights = weights[ptr:ptr + num_params].reshape(
            (shape[3], shape[2], shape[0], shape[1]))
        var_weights = np.transpose(var_weights, (2, 3, 1, 0))
        ptr += num_params
        assign_ops.append(tf.assign(conv_var, var_weights))

    return assign_ops

## 6. Download Weights & Class Names

Run the cell below to download the YOLOv3 weights (~236 MB) and COCO class names if they are not already present.

In [12]:
import os
import urllib.request

os.makedirs('weights', exist_ok=True)

WEIGHTS_URL = 'https://pjreddie.com/media/files/yolov3.weights'
WEIGHTS_PATH = 'weights/yolov3.weights'
NAMES_URL = 'https://raw.githubusercontent.com/pjreddie/darknet/master/data/coco.names'
NAMES_PATH = 'weights/coco.names'

if not os.path.exists(WEIGHTS_PATH):
    print('Downloading YOLOv3 weights (~236 MB)...')
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_PATH)
    print('Done.')
else:
    print('Weights already present.')

if not os.path.exists(NAMES_PATH):
    print('Downloading COCO class names...')
    urllib.request.urlretrieve(NAMES_URL, NAMES_PATH)
    print('Done.')
else:
    print('Class names already present.')

HTTPError: HTTP Error 403: Forbidden

## 7. Running the Model

Place your test images in the `images/` folder and update `img_names` below.

In [ ]:
import os

os.makedirs('images', exist_ok=True)

# Update this list with your image paths
img_names = [
    'images/dog.jpg',
    'images/office.jpg',
]

# Preview input images
for img_path in img_names:
    if os.path.exists(img_path):
        display(Image.open(img_path))
    else:
        print(f'Image not found: {img_path}')

In [ ]:
# Filter to existing images only
img_names = [p for p in img_names if os.path.exists(p)]
assert len(img_names) > 0, 'No valid images found. Add images to the images/ folder.'

batch_size = len(img_names)
batch = load_images(img_names, model_size=_MODEL_SIZE)
class_names = load_class_names(NAMES_PATH)
n_classes = len(class_names)

max_output_size = 10
iou_threshold = 0.5
confidence_threshold = 0.5

# Reset graph for clean state
tf.reset_default_graph()

model = Yolo_v3(
    n_classes=n_classes,
    model_size=_MODEL_SIZE,
    max_output_size=max_output_size,
    iou_threshold=iou_threshold,
    confidence_threshold=confidence_threshold
)

inputs = tf.placeholder(tf.float32, [batch_size, *_MODEL_SIZE, 3])
detections = model(inputs, training=False)

model_vars = tf.global_variables(scope='yolo_v3_model')
assign_ops = load_weights(model_vars, WEIGHTS_PATH)

with tf.Session() as sess:
    sess.run(assign_ops)
    print('Weights loaded successfully.')
    detection_result = sess.run(detections, feed_dict={inputs: batch})

print('Detection complete. Drawing boxes...')
draw_boxes(img_names, detection_result, class_names, _MODEL_SIZE)

## 8. Video Processing

Run detection frame-by-frame on a local video file. Output is saved to `output_video.avi`.

In [ ]:
def process_video(video_path, output_path, class_names, model_size,
                  iou_threshold=0.5, confidence_threshold=0.5,
                  max_output_size=10):
    """Runs YOLOv3 detection on every frame of a video and writes annotated output.

    Args:
        video_path: Path to input video file.
        output_path: Path for annotated output video.
        class_names: List of class name strings.
        model_size: (H, W) of model input.
        iou_threshold: IoU threshold for NMS.
        confidence_threshold: Min confidence for a detection.
        max_output_size: Max boxes per class per frame.
    """
    n_classes = len(class_names)
    colors = ((np.array(color_palette('hls', 80)) * 255)).astype(np.uint8)

    tf.reset_default_graph()
    model = Yolo_v3(
        n_classes=n_classes,
        model_size=model_size,
        max_output_size=max_output_size,
        iou_threshold=iou_threshold,
        confidence_threshold=confidence_threshold
    )

    inputs = tf.placeholder(tf.float32, [1, *model_size, 3])
    detections = model(inputs, training=False)
    model_vars = tf.global_variables(scope='yolo_v3_model')
    assign_ops = load_weights(model_vars, WEIGHTS_PATH)

    cap = cv2.VideoCapture(video_path)
    frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter(output_path, fourcc, fps, (frame_w, frame_h))

    resize_factor = (frame_w / model_size[0], frame_h / model_size[1])

    with tf.Session() as sess:
        sess.run(assign_ops)
        print('Weights loaded. Processing video...')
        frame_count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            resized = cv2.resize(rgb, model_size)
            batch = np.expand_dims(resized.astype(np.float32), axis=0)

            result = sess.run(detections, feed_dict={inputs: batch})
            boxes_dict = result[0]

            for cls in range(n_classes):
                boxes = boxes_dict.get(cls)
                if boxes is None or len(boxes) == 0:
                    continue
                color = tuple(int(c) for c in colors[cls])
                for box in boxes:
                    xy, conf = box[:4], box[4]
                    x1 = int(xy[0] * resize_factor[0])
                    y1 = int(xy[1] * resize_factor[1])
                    x2 = int(xy[2] * resize_factor[0])
                    y2 = int(xy[3] * resize_factor[1])
                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                    label = f'{class_names[cls]} {conf:.0%}'
                    cv2.putText(frame, label, (x1, max(y1 - 5, 0)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1,
                                cv2.LINE_AA)

            out.write(frame)
            frame_count += 1
            if frame_count % 30 == 0:
                print(f'  Processed {frame_count} frames...')

    cap.release()
    out.release()
    print(f'Video saved to {output_path} ({frame_count} frames)')


# Example usage — update VIDEO_PATH to your video file
# process_video(
#     video_path='images/input_video.mp4',
#     output_path='images/output_video.avi',
#     class_names=class_names,
#     model_size=_MODEL_SIZE
# )

## 9. Acknowledgements

- [YOLOv3 Official Paper](https://pjreddie.com/media/files/papers/YOLOv3.pdf) — Redmon & Farhadi (2018)
- [Darknet Reference Implementation](https://github.com/pjreddie/darknet)
- TensorFlow ResNet reference implementation (fixed padding, data format conventions)
- COCO Dataset — 80 object categories including common household objects